# Quick Start Guide
github link: [cytozip](https://github.com/DingWB/cytozip)

## 1. Installation

### conda
https://anaconda.org/bioconda/cytozip
```shell
mamba install -c wubinding -c bioconda cytozip
# or
conda install -c wubinding -c bioconda cytozip
# "-c wubinding" is optional, which may be the latest version
```

### pip
```shell
# Prerequisites (one of):
conda install -c bioconda htslib libdeflate samtools         # recommended
# apt-get install libhts-dev libdeflate-dev samtools          # Debian/Ubuntu
# brew install htslib libdeflate samtools                     # macOS
pip install cytozip
# or reinstall
pip uninstall -y cytozip && pip install git+http://github.com/DingWB/cytozip
```

In [1]:
# Make sure czip is installed:
!which czip

~/Software/conda/m3c/bin/czip


In [2]:
import os
os.chdir(os.path.expanduser("~/Projects/test_cytozip"))

## 2. Build reference cz file from a reference genome fasta file
The reference holds the genome-wide (chrom, pos, strand, context) axis. Per-cell .cz then store only mc/cov and reuse the reference’s positions, cutting per-cell size by ~5×.

In [10]:
! time czip build_ref --genome ~/Ref/hg38/hg38_ucsc_with_chrL.fa \
                      --output cytozip_example_data/hg38_with_chrL.allc.cz \
                      --jobs 24 \
                      --chroms ~/Ref/hg38/hg38_ucsc_with_chrL.main.chrom.sizes
# --genome reference genome
# --chroms Optional. A .fai or text file whose first column lists the chromosomes to restrict analysis to (typically the main chromosomes).
# hg38_ucsc_with_chrL.main.chrom.sizes can be downloaded from https://figshare.com/ndownloader/files/67164602

2026-07-31 20:38:29.715 | DEBUG    | cytozip.allc:WriteC:68 - chr1
2026-07-31 20:38:30.345 | DEBUG    | cytozip.allc:WriteC:68 - chr10
2026-07-31 20:38:31.093 | DEBUG    | cytozip.allc:WriteC:68 - chr11
2026-07-31 20:38:31.794 | DEBUG    | cytozip.allc:WriteC:68 - chr12
2026-07-31 20:38:32.381 | DEBUG    | cytozip.allc:WriteC:68 - chr13
2026-07-31 20:38:32.967 | DEBUG    | cytozip.allc:WriteC:68 - chr14
2026-07-31 20:38:33.512 | DEBUG    | cytozip.allc:WriteC:68 - chr15
2026-07-31 20:38:33.981 | DEBUG    | cytozip.allc:WriteC:68 - chr16
2026-07-31 20:38:34.431 | DEBUG    | cytozip.allc:WriteC:68 - chr17
2026-07-31 20:38:34.861 | DEBUG    | cytozip.allc:WriteC:68 - chr18
2026-07-31 20:38:35.193 | DEBUG    | cytozip.allc:WriteC:68 - chr19
2026-07-31 20:38:36.629 | DEBUG    | cytozip.allc:WriteC:68 - chr2
2026-07-31 20:38:36.852 | DEBUG    | cytozip.allc:WriteC:68 - chr20
2026-07-31 20:38:37.088 | DEBUG    | cytozip.allc:WriteC:68 - chr21
2026-07-31 20:38:37.366 | DEBUG    | cytozip.allc:

The reference .cz file (hg38_with_chrL.allc.cz) can also be downloaded from figshare: https://figshare.com/ndownloader/files/67164455
<br/>
Or directly download it on linux with pyfigshare:
```shell
pip install pyfigshare
figshare download 32095567 --file_id 66985196 --outdir cytozip_example_data
```

In [11]:
!czip header -I cytozip_example_data/hg38_with_chrL.allc.cz

magic  :  b'CZIP'
version  :  0.39
total_size  :  1430352768
message  :  /home/x-wding2/Ref/hg38/hg38_ucsc_with_chrL.fa
formats  :  ['Q', 'c', '3s']
columns  :  ['pos', 'strand', 'context']
sort_col  :  0
delta_cols  :  [0]
chunk_dims  :  ['chrom']
header_size  :  102


In [12]:
! czip view -I cytozip_example_data/hg38_with_chrL.allc.cz --show_dims 0 | head

chrom	pos	strand	context
chr1	10004	+	CCC
chr1	10005	+	CCT
chr1	10006	+	CTA
chr1	10010	+	CCC
chr1	10011	+	CCT
chr1	10012	+	CTA
chr1	10016	+	CCC
chr1	10017	+	CCT
chr1	10018	+	CTA


## 3. Call methylation from bam file and save as .cz format

### 2.1 Download example bam files

```shell
# Download example bam files from figshare with your browser: https://figshare.com/ndownloader/files/63998524
# or download with command line using pyfigshare:
pip install pyfigshare # https://github.com/DingWB/pyfigshare
figshare download 32095567 --file_id 66902144,66902129,66902816,66902822 --outdir cytozip_example_data
```

### 2.2 Call DNA methylation using cytozip

In [10]:
! time czip bam_to_cz --input cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.bam \
               --genome ~/Ref/hg38/hg38_ucsc_with_chrL.fa \
               --output cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz \
               --reference cytozip_example_data/hg38_with_chrL.allc.cz \
               --chroms ~/Ref/hg38/hg38_ucsc_with_chrL.main.chrom.sizes

[W::hts_idx_load3] The index file is older than the data file: cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.bam.bai

real	1m33.907s
user	1m21.110s
sys	0m9.285s


Python API
```python
from cytozip.bam import bam_to_cz
bam_to_cz(bam_path="cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.bam", \
                  genome="~/Ref/hg38/hg38_ucsc_with_chrL.fa", \
                  output="cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz", \
                  reference="~/Ref/hg38/hg38_with_chrL.allc.cz",
                  chroms="~/Ref/hg38/hg38_ucsc_with_chrL.main.chrom.sizes")
```

## 4. Convert allc/bed/stdin to .cz

### 4.1 convert stream input to cz with pipe

In [13]:
! czip tocz -h

usage: czip tocz [-h] -O OUTPUT [-I INPUT] [-F FORMATS] [-C COLUMNS]
                 [-D CHUNK_DIMS] [-u USECOLS] [-d KEY_COLS] [-s SEP]
                 [-c BATCH_SIZE] [--header HEADER] [--skiprows SKIPROWS]
                 [-m MESSAGE] [-l LEVEL] [--delta_cols DELTA_COLS]

options:
  -h, --help            show this help message and exit
  -O OUTPUT, --output OUTPUT
                        output .cz file (default: None)
  -I INPUT, --input INPUT
                        input file (stdin if omitted) (default: None)
  -F FORMATS, --formats FORMATS
                        column formats, comma-separated (struct chars).
                        Unsigned ints cap values: B=255, H=65535, I=2^32-1,
                        Q=2^64-1; larger values are truncated (saturated) to
                        the max. Default B for single-cell mc/cov saves space
                        and is safe: counts >255 are usually repeat-region
                        artifacts and downstream ALLCools DMR cli

In [14]:
! zcat cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.allc.tsv.gz | head

chr1	14932	-	CTT	0	1	1
chr1	14933	-	CCT	0	1	1
chr1	14935	-	CAC	1	1	1
chr1	14938	-	CAG	1	1	1
chr1	14939	-	CCA	0	1	1
chr1	14944	-	CTG	0	1	1
chr1	14945	-	CCT	0	1	1
chr1	14946	-	CCC	0	1	1
chr1	14948	-	CGC	1	1	1
chr1	14949	-	CCG	0	1	1

gzip: stdout: Broken pipe


In [10]:
# write chrom, position, mc and cov into .cz file with DEFLATE compress
! zcat cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.allc.tsv.gz | czip tocz \
        --output cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz \
        --formats Q,B,B --usecols 1,4,5 --columns pos,mc,cov \
        --delta_cols pos # file size would be smaller with DEFLATE compress: 47M vs 29M

In [11]:
! czip view -I cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz --show_dims 0 | head

chrom	pos	mc	cov
chr1	14932	0	1
chr1	14933	0	1
chr1	14935	1	1
chr1	14938	1	1
chr1	14939	0	1
chr1	14944	0	1
chr1	14945	0	1
chr1	14946	0	1
chr1	14948	1	1


### 4.2 allc to cz with reference cz as input

In [15]:
! czip allc2cz -h

usage: czip allc2cz [-h] -I INPUT -O OUTPUT [-r REFERENCE]
                    [--missing_value MISSING_VALUE] [-F FORMATS] [-C COLUMNS]
                    [-D CHUNK_DIMS] [-u USECOLS] [--ref_pos_col REF_POS_COL]
                    [--allc_pos_col ALLC_POS_COL] [-s SEP] [--chroms CHROMS]
                    [-c BATCH_SIZE] [--sort_col SORT_COL]
                    [--delta_cols DELTA_COLS] [-j JOBS] [--pattern PATTERN]
                    [--no_skip_existing]

options:
  -h, --help            show this help message and exit
  -I INPUT, --input INPUT
                        input allc.tsv.gz, OR a directory containing many
                        allc.tsv.gz (batch mode: --output must be a directory)
                        (default: None)
  -O OUTPUT, --output OUTPUT
                        output .cz file (single-file), or output directory
                        (batch mode) (default: None)
  -r REFERENCE, --reference REFERENCE
                        reference .cz file (default: N

In [14]:
! rm -f cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz
! time czip allc2cz --input cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.allc.tsv.gz \
                    --output cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz \
                    --reference cytozip_example_data/hg38_with_chrL.allc.cz \
                    --formats "B,B" # for single cell, B is fine (max is 255), please change to H for pseudobulk data

2026-07-22 14:30:42.028 | INFO     | cytozip.allc:allc2cz:348 - /anvil/projects/x-mcb130189/Wubin/test_cytozip/cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.allc.tsv.gz

real	0m42.276s
user	0m35.840s
sys	0m4.946s


In [16]:
! ls cytozip_example_data/hg38_allc -sh

total 121M
 97M UWA7648_CX1819_NAC_1_P10-1-K18-A10.allc.tsv.gz
2.0M UWA7648_CX1819_NAC_1_P10-1-K18-A10.allc.tsv.gz.tbi
 22M UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz


In [19]:
# view
! czip view -I cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz --show_dims 0 | head

chrom	mc	cov
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0


In [21]:
# view cz file with coordinates (reference cz)
! czip view -I cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz \
            --show_dims 0 \
            -r cytozip_example_data/hg38_with_chrL.allc.cz | head

chrom	pos	strand	context	mc	cov
chr1	10004	+	CCC	0	0
chr1	10005	+	CCT	0	0
chr1	10006	+	CTA	0	0
chr1	10010	+	CCC	0	0
chr1	10011	+	CCT	0	0
chr1	10012	+	CTA	0	0
chr1	10016	+	CCC	0	0
chr1	10017	+	CCT	0	0
chr1	10018	+	CTA	0	0


In [27]:
# query cz file
! time czip query -I cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz \
            --chunk_key chr1 --start 14932 --end 14949 \
            -r cytozip_example_data/hg38_with_chrL.allc.cz

chrom	pos	strand	context	mc	cov
chr1	14932	-	CTT	0	1
chr1	14933	-	CCT	0	1
chr1	14935	-	CAC	1	1
chr1	14936	+	CTG	0	0
chr1	14938	-	CAG	1	1
chr1	14939	-	CCA	0	1
chr1	14940	+	CCC	0	0
chr1	14941	+	CCA	0	0
chr1	14942	+	CAG	0	0
chr1	14944	-	CTG	0	1
chr1	14945	-	CCT	0	1
chr1	14946	-	CCC	0	1
chr1	14947	+	CGG	0	0
chr1	14948	-	CGC	1	1
chr1	14949	-	CCG	0	1

real	0m0.251s
user	0m0.136s
sys	0m0.097s


## 5. Python API

In [3]:
from cytozip import Reader
reader=Reader("cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz")
print(reader.header)
df_chunks=reader.summary_chunks(printout=False)
print(df_chunks.head())

{'magic': b'CZIP', 'version': 0.36, 'total_size': 22810155, 'message': 'hg38_with_chrL.allc.cz', 'formats': ['B', 'B'], 'columns': ['mc', 'cov'], 'sort_col': None, 'delta_cols': [], 'chunk_dims': ['chrom'], 'header_size': 61}
           chrom  chunk_start_offset  chunk_size  chunk_tail_offset  \
chunk_dims                                                            
(chr1,)     chr1                  61     1906076            1912030   
(chr2,)     chr2             1912030     1912984            3830947   
(chr3,)     chr3             3830947     1541226            5376994   
(chr4,)     chr4             5376994     1443646            6825093   
(chr5,)     chr5             6825093     1404376            8233866   

            chunk_nblocks  chunk_nrows  
chunk_dims                              
(chr1,)               734     96166571  
(chr2,)               739     96769083  
(chr3,)               600     78577742  
(chr4,)               554     72568001  
(chr5,)               547     

In [4]:
reader.chunk_key2offset

{('chr1',): 61,
 ('chr2',): 1912030,
 ('chr3',): 3830947,
 ('chr4',): 5376994,
 ('chr5',): 6825093,
 ('chr6',): 8233866,
 ('chr7',): 9558673,
 ('chr8',): 10822796,
 ('chr9',): 11951699,
 ('chr10',): 12908810,
 ('chr11',): 14011574,
 ('chr12',): 15114383,
 ('chr13',): 16196205,
 ('chr14',): 17003787,
 ('chr15',): 17696549,
 ('chr16',): 18349412,
 ('chr17',): 19030001,
 ('chr18',): 19690263,
 ('chr19',): 20347873,
 ('chr20',): 20822410,
 ('chr21',): 21390273,
 ('chr22',): 21689571,
 ('chrX',): 22001315,
 ('chrY',): 22701140,
 ('chrM',): 22800785,
 ('chrL',): 22801381}

In [5]:
data=reader.chunk2numpy(dims=('chr1',))
data['f0'],data['f1'] # mc and cov

(array([0, 0, 0, ..., 0, 0, 0], dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], dtype=uint8))

In [6]:
df=reader.chunk2df(dims=('chr1',))
df.head()

,mc,cov
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0


In [8]:
r=reader.query(chunk_key=('chr1',),start=14932,end=14949,printout=False,
                    reference="cytozip_example_data/hg38_with_chrL.allc.cz")
#r.__next__()
for record in r:
    print(record)

['chr1', '14932', '-', 'CTT', '0', '1']
['chr1', '14933', '-', 'CCT', '0', '1']
['chr1', '14935', '-', 'CAC', '1', '1']
['chr1', '14936', '+', 'CTG', '0', '0']
['chr1', '14938', '-', 'CAG', '1', '1']
['chr1', '14939', '-', 'CCA', '0', '1']
['chr1', '14940', '+', 'CCC', '0', '0']
['chr1', '14941', '+', 'CCA', '0', '0']
['chr1', '14942', '+', 'CAG', '0', '0']
['chr1', '14944', '-', 'CTG', '0', '1']
['chr1', '14945', '-', 'CCT', '0', '1']
['chr1', '14946', '-', 'CCC', '0', '1']
['chr1', '14947', '+', 'CGG', '0', '0']
['chr1', '14948', '-', 'CGC', '1', '1']
['chr1', '14949', '-', 'CCG', '0', '1']


In [9]:
data=reader.query_numpy(chunk_key=('chr1',),start=14932,end=14949,
                    reference="cytozip_example_data/hg38_with_chrL.allc.cz")
data

(array([14932, 14933, 14935, 14936, 14938, 14939, 14940, 14941, 14942,
        14944, 14945, 14946, 14947, 14948, 14949], dtype=uint64),
 array([(0, 1), (0, 1), (1, 1), (0, 0), (1, 1), (0, 1), (0, 0), (0, 0),
        (0, 0), (0, 1), (0, 1), (0, 1), (0, 0), (1, 1), (0, 1)],
       dtype=[('f0', 'u1'), ('f1', 'u1')]))

In [10]:
reader.close()

## 6. Remote Reading (from URL)
czip supports reading .cz files directly from HTTP/HTTPS URLs without downloading the entire file. This uses HTTP Range requests and a chunk index for O(1) lookup.

In [17]:
! time czip query -I https://neomorph.salk.edu/ftp/bican/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz \
            --chunk_key chr1 --start 14932 --end 14949 \
            -r https://neomorph.salk.edu/ftp/cz/hg38_with_chrL.allc.cz

chrom	pos	strand	context	mc	cov
chr1	14932	-	CTT	0	1
chr1	14933	-	CCT	0	1
chr1	14935	-	CAC	1	1
chr1	14936	+	CTG	0	0
chr1	14938	-	CAG	1	1
chr1	14939	-	CCA	0	1
chr1	14940	+	CCC	0	0
chr1	14941	+	CCA	0	0
chr1	14942	+	CAG	0	0
chr1	14944	-	CTG	0	1
chr1	14945	-	CCT	0	1
chr1	14946	-	CCC	0	1
chr1	14947	+	CGG	0	0
chr1	14948	-	CGC	1	1
chr1	14949	-	CCG	0	1

real	0m2.198s
user	0m0.228s
sys	0m0.111s


In [ ]:
import cytozip as czip

# Open a remote .cz file via URL
# url = "https://example.com/path/to/file.cz"
# reader = czip.Reader.from_url(url)
# reader.print_header()

# Or simply pass a URL to Reader (auto-detected)
# reader = czip.Reader(url)
# for record in reader.fetch(("chr1",)):
#     print(record)
# reader.close()
print("Remote reading requires a .cz file hosted on an HTTP server supporting Range requests.")

## 7. Convert cz to allc & compare cz with allc

### 7.1 Download example paired cz and allc files from figshare

Download example paired allc and cz files from figshare: <br/>
https://figshare.com/articles/dataset/cytozip_example_data/32095567<br/>
or download with command line using pyfigshare:
```shell
pip install pyfigshare # https://github.com/DingWB/pyfigshare
figshare download 32095567 --cpu 4 --outdir cytozip_example_data # download the whole folder (cytozip_example_data)
```

### 7.2 Convert .cz to .allc.tsv.gz

In [20]:
! czip header -I cytozip_example_data/hg38_cz/UWA7648_CX1819_NAC_1_P9-1-M14-C14.cz

magic  :  b'CZIP'
version  :  0.36
total_size  :  55865977
message  :  hg38_with_chrL.allc.cz
formats  :  ['B', 'B']
columns  :  ['mc', 'cov']
sort_col  :  None
delta_cols  :  []
chunk_dims  :  ['chrom']
header_size  :  61


In [21]:
# Convert .cz to .allc.tsv.gz
! time czip to_bgzip -I cytozip_example_data/hg38_cz/UWA7648_CX1819_NAC_1_P9-1-M14-C14.cz \
                -O cytozip_example_data/hg38_cz/UWA7648_CX1819_NAC_1_P9-1-M14-C14.allc.tsv.gz \
                -r cytozip_example_data/hg38_with_chrL.allc.cz --cov_col cov --allc_format


real	2m44.037s
user	2m34.460s
sys	0m8.519s


### 7.3 Directly compare .cz and ALLCools generated .allc.tsv.gz

In [24]:
# validate whether cz and allc store the same values in Python
import pandas as pd
df_allc=pd.read_csv("cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P9-1-M14-C14.allc.tsv.gz", 
                    sep="\t", header=None, usecols=[0, 1,2,3,4, 5],names=["chrom", "pos", 'strand','context',"mc_allc", "cov_allc"],
    )
df_allc.set_index(['chrom','pos','strand','context'],inplace=True)
df_allc

mc_allc  cov_allc
chrom pos   strand context                   
chr1  32639 +      CAC            1         1
      32641 +      CCC            1         1
      32642 +      CCA            1         1
      32643 +      CAT            1         1
      32646 +      CGG            1         1
...                             ...       ...
chrL  48487 -      CAC            0         3
      48492 -      CGG            2         3
      48496 -      CTG            0         3
      48497 -      CCT            0         3
      48502 -      CGT            2         3

[73852777 rows x 2 columns]

In [25]:
from cytozip import Reader
cell = Reader("cytozip_example_data/hg38_cz/UWA7648_CX1819_NAC_1_P9-1-M14-C14.cz")
df_cz=cell.to_df(reference="cytozip_example_data/hg38_with_chrL.allc.cz")
df_cz.rename(columns={'mc':'mc_cz','cov':'cov_cz'},inplace=True)
df_cz.set_index(['chrom','pos','strand','context'],inplace=True)
df_cz

mc_cz  cov_cz
chrom pos   strand context               
chr1  32639 +      CAC          1       1
      32641 +      CCC          1       1
      32642 +      CCA          1       1
      32643 +      CAT          1       1
      32646 +      CGG          1       1
...                           ...     ...
chrL  48487 -      CAC          0       3
      48492 -      CGG          2       3
      48496 -      CTG          0       3
      48497 -      CCT          0       3
      48502 -      CGT          2       3

[73852777 rows x 2 columns]

In [26]:
df_combined=pd.concat([df_allc,df_cz],axis=1)
print(df_combined.head())
df_diff=df_combined.loc[(df_combined.mc_allc != df_combined.mc_cz) | (df_combined.cov_allc != df_combined.cov_cz)]
print(f"{df_diff.shape[0]} mismatch between allc and cz")
df_diff

                            mc_allc  cov_allc  mc_cz  cov_cz
chrom pos   strand context                                  
chr1  32639 +      CAC            1         1      1       1
      32641 +      CCC            1         1      1       1
      32642 +      CCA            1         1      1       1
      32643 +      CAT            1         1      1       1
      32646 +      CGG            1         1      1       1
25 mismatch between allc and cz


mc_allc  cov_allc  mc_cz  cov_cz
chrom pos       strand context                                  
chr1  125180207 -      CAA          195       335    195     255
      125180208 -      CCA          145       336    145     255
      125180210 -      CAC          205       346    205     255
      125180213 -      CAA          197       340    197     255
      125180223 -      CGA          296       322    255     255
      125180233 -      CGA          294       319    255     255
      125180236 -      CAT          184       328    184     255
      125180249 -      CGA          295       309    255     255
      125180257 -      CAT          164       313    164     255
      125180259 -      CGC          289       309    255     255
      125180262 -      CAT          166       320    166     255
      125180282 -      CTA          182       352    182     255
      125180285 -      CTT          183       356    183     255
      125180288 -      CAT          185       349    185     255
      125180298 -      CGA          300       334    255     255
      125180300 -      CTC          171       341    171     255
      125180308 -      CGA          312       328    255     255
      125180311 -      CAT          175       333    175     255
      125180315 -      CAA          172       329    172     255
      125180330 -      CGA          299       314    255     255
      125180333 -      CAT          140       278    140     255
      125180336 -      CGT          244       264    244     255
      125180352 -      CAG          138       259    138     255
      125180359 -      CAT          147       273    147     255
      125180369 -      CTA          142       262    142     255

All mismatches arise because cytozip clips mc and cov to a maximum of 255, since single-cell data is stored using the 1-byte unsigned integer format (B) by default.

### 7.4 Compare .cz converted .allc.tsv.gz and ALLCools generated .allc.tsv.gz

In [22]:
! czip compare_allc -h

usage: czip compare_allc [-h] -a ALLC1 -b ALLC2 [--keep_zero_cov]
                         [--dtype {B,H,I,Q,None}] [-O OUTPUT] [--sep SEP]

options:
  -h, --help            show this help message and exit
  -a ALLC1, --allc1 ALLC1
                        first allc.tsv[.gz] file (default: None)
  -b ALLC2, --allc2 ALLC2
                        second allc.tsv[.gz] file (default: None)
  --keep_zero_cov       keep records with cov==0 (default: drop them)
                        (default: False)
  --dtype {B,H,I,Q,None}
                        unsigned-int format to derive mc/cov clamp bound
                        (B=255, H=65535, I=2^32-1, Q=2^64-1; None=disable
                        clamping; default: B). With B, mc/cov >255 are
                        truncated to 255 to mirror how cytozip packs single-
                        cell mc/cov as 1-byte B; counts >255 are usually
                        repeat-region artifacts and downstream ALLCools DMR
                        clips c

In [23]:
! czip compare_allc -a cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P9-1-M14-C14.allc.tsv.gz \
                    -b cytozip_example_data/hg38_cz/UWA7648_CX1819_NAC_1_P9-1-M14-C14.allc.tsv.gz

2026-07-26 01:38:46.888 | INFO     | cytozip.allc:compare_allc:889 - compare_allc: 73852777 records in allc1, 73852777 in allc2; only_in_1=0, only_in_2=0, mc_diff=0, cov_diff=0, total_diff=0


In [28]:
! czip compare_allc -a cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P9-1-M14-C14.allc.tsv.gz \
                    -b cytozip_example_data/hg38_cz/UWA7648_CX1819_NAC_1_P9-1-M14-C14.allc.tsv.gz --dtype None
# add --dtype=None to show the difference caused by truncation of 255

2026-07-26 01:44:29.885 | INFO     | cytozip.allc:compare_allc:889 - compare_allc: 73852777 records in allc1, 73852777 in allc2; only_in_1=0, only_in_2=0, mc_diff=7, cov_diff=25, total_diff=25
